In [6]:
DATASET_PATH = "../datasets"

**M57-patent dataset**

Converting m57 desktop image xmls into csv

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import os

def xml_to_raw_csv(xml_files, output_csv="../datasets/m57_raw_dump.csv"):
    all_rows = []
    
    for xml_file in xml_files:
        if not os.path.exists(xml_file):
            print(f"⚠️ Skipping missing file: {xml_file}")
            continue
            
        print(f"Parsing {xml_file}...")
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            source_user = os.path.basename(xml_file).split('-')[0]

            for file_obj in root.findall(".//fileobject"):
                # --- 1. OPTIMIZATION: Filter Directories Immediately ---
                ntype = file_obj.find("name_type")
                
                # 'd' = Directory. We skip these to save massive space.
                if ntype is not None and ntype.text == 'd':
                    continue

                row = {}
                row['source_user'] = source_user
                row['type'] = ntype.text if ntype is not None else "unknown"

                # --- 2. Extract Data ---
                
                # Filename (Raw)
                fname_tag = file_obj.find("filename")
                if fname_tag is not None:
                    row['raw_path'] = fname_tag.text
                else:
                    continue # Skip objects with no name

                # File Size
                fsize = file_obj.find("filesize")
                row['size'] = int(fsize.text) if fsize is not None else 0

                # Timestamps
                mtime = file_obj.find("mtime")
                row['mtime_iso'] = mtime.text if mtime is not None else None
                
                # Flags ('unalloc' = 1 means DELETED file)
                unalloc = file_obj.find("unalloc")
                row['is_deleted'] = 1 if (unalloc is not None and unalloc.text == '1') else 0

                all_rows.append(row)
                
        except Exception as e:
            print(f"❌ Error parsing {xml_file}: {e}")

    # Convert to DataFrame
    df = pd.DataFrame(all_rows)
    
    # Save raw output with BOM for VS Code compatibility
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n✅ Success! Saved {len(df)} file rows (no directories) to '{output_csv}'")
    return df

# --- CONFIGURATION ---
target_files = [
    "../datasets/charlie-2009-12-11.xml",
    "../datasets/jo-2009-12-11-002.xml",
    "../datasets/pat-2009-12-11.xml",
    "../datasets/terry-2009-12-11-002.xml"
]

# RUN IT
xml_to_raw_csv(target_files)

Parsing ../datasets/charlie-2009-12-11.xml...
Parsing ../datasets/jo-2009-12-11-002.xml...
Parsing ../datasets/pat-2009-12-11.xml...
Parsing ../datasets/terry-2009-12-11-002.xml...

✅ Success! Saved 207461 file rows (no directories) to '../datasets/m57_raw_dump.csv'


,source_user,type,raw_path,size,mtime_iso,is_deleted
0,charlie,r,$AttrDef,2560,2009-11-08T16:58:56Z,0
1,charlie,r,$AVG/$CHJW/23327068-982b-4c1f-a43a-fb5e73e62883,384,2009-11-10T00:45:33Z,0
2,charlie,r,$AVG/$CHJW/3d1d098e-5278-4184-b6c1-681170b37870,328,2009-11-10T00:45:33Z,0
3,charlie,r,$AVG/$CHJW/avgcchff.dat,934072,2009-12-09T10:54:18Z,0
4,charlie,r,$AVG/$CHJW/avgcchfi.dat,118600,2009-12-11T00:53:57Z,0
...,...,...,...,...,...,...
207456,terry,-,$OrphanFiles/package_101_for_kb976098_bf~31bf3...,8461,2009-10-30T12:54:05Z,1
207457,terry,-,$OrphanFiles/package_100_for_kb976098~31bf3856...,11964,2009-10-30T10:41:08Z,1
207458,terry,-,$OrphanFiles/package_100_for_kb976098~31bf3856...,9339,2009-10-30T11:38:56Z,1
207459,terry,-,$OrphanFiles/package_100_for_kb976098_bf~31bf3...,1978,2009-10-30T10:50:11Z,1


cleaning and modernizing m57 data

In [ ]:
import pandas as pd
import numpy as np
import datetime
import time
import random


def clean_and_modernize(input_csv, output_csv):
    print(f"Loading {input_csv}...")
    df = pd.read_csv(input_csv)
    initial_len = len(df)

    # --- 1. FILTERING (Forensic Noise) ---
    # Keep only existing files.
    # 'r' = Regular File. '-' often appears in forensic tools for files too, so we keep those.
    # We drop 'd' (directories) if any slipped through.
    df = df[df["type"].isin(["r", "-"])]

    # Remove deleted files
    df = df[df["is_deleted"] == 0]

    # Remove NTFS Metadata (Files starting with $)
    df = df[~df["raw_path"].str.startswith("$")]

    # # Remove "System Volume Information"
    # df = df[~df["raw_path"].str.contains("System Volume Information", case=False)]

    print(f"Filtered {initial_len} -> {len(df)} valid files.")

    # --- 2. MODERNIZATION LOGIC ---
    TIME_SHIFT = 15 * 365 * 24 * 3600  # Shift 2009 -> 2024
    current_unix = int(time.time())

    def process_row(row):
        raw_path = str(row["raw_path"])

        # A. Fix Path Separators & Drive
        # Convert "Documents/Resume.doc" -> "C:\Documents\Resume.doc"
        win_path = "C:\\" + raw_path.replace("/", "\\")

        # B. Modernize "Documents and Settings" -> "Users"
        if "Documents and Settings" in win_path:
            win_path = win_path.replace("Documents and Settings", "Users")

        # C. Inject Cloud Context (10% chance for user docs)
        if "Users" in win_path and "Documents" in win_path and random.random() < 0.10:
            win_path = win_path.replace("Documents", "OneDrive\\Documents")

        # Split into directory and filename
        if "\\" in win_path:
            path_dir, name = win_path.rsplit("\\", 1)
        else:
            path_dir = "C:\\"
            name = win_path

        # D. Modernize Extensions
        if name.endswith(".doc") and random.random() > 0.2:
            name += "x"
        elif name.endswith(".xls") and random.random() > 0.2:
            name += "x"
        elif name.endswith(".ppt") and random.random() > 0.2:
            name += "x"

        # E. Parse Timestamp
        try:
            if pd.notnull(row["mtime_iso"]):
                dt = datetime.datetime.fromisoformat(
                    str(row["mtime_iso"]).replace("Z", "+00:00")
                )
                unix_time = int(dt.timestamp()) + TIME_SHIFT
            else:
                unix_time = current_unix - random.randint(
                    0, 1000000
                )  # Compute missing dates
        except:
            unix_time = current_unix

        # Extract extension
        ext = "." + name.rsplit(".", 1)[-1].lower() if "." in name else ""

        return pd.Series(
            [name, path_dir, ext, row["size"], unix_time, row["source_user"]]
        )

    print("Modernizing paths and dates (this may take a moment)...")
    new_cols = [
        "filename",
        "path",
        "extension",
        "size_bytes",
        "mod_time_unix",
        "source_user",
    ]
    df[new_cols] = df.apply(process_row, axis=1)

    # --- 3. DOWNSAMPLE SYSTEM NOISE ---
    # Downsample 'System32' and 'DLLs' by 90%
    system_mask = (df["path"].str.lower().str.contains("system32")) | (
        df["extension"] == ".dll"
    )
    # Keep 10% of system files, keep 100% of everything else
    keep_mask = ~system_mask | (np.random.rand(len(df)) < 0.10)
    df = df[keep_mask]

    # Save final columns
    df[new_cols].to_csv(output_csv, index=False)
    print(f"✅ Training data saved to '{output_csv}' ({len(df)} rows)")


clean_and_modernize(
    "../datasets/m57_raw_dump.csv", "../datasets/m57_modernized_dataset.csv"
)

Loading ../datasets/m57_raw_dump.csv...
Filtered 207461 -> 182140 valid files.
Modernizing paths and dates (this may take a moment)...
✅ Training data saved to '../datasets/m57_modernized_dataset.csv' (136846 rows)


**GovDocs Dataset**

In [ ]:
import pandas as pd
import datasets

# Replace with the path to the file you just downloaded
parquet_file = "../datasets/train-00000-of-00001.parquet"

print("Reading Parquet file...")
df = pd.read_parquet(parquet_file)

# Save as CSV
output_csv = "../datasets/govdocs_metadata.csv"
df.to_csv(output_csv, index=False)

print(f"✅ Converted! Saved {len(df)} rows to '{output_csv}'")
df

c:\Users\User\Documents\university\SemesterE\CyberSecurityMLproject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading Parquet file...
✅ Converted! Saved 229917 rows to '../datasets/govdocs_metadata.csv'
     filename                                              title  \
0  004901.pdf                                               None   
1  388027.pdf                                Chapter 7 pp121-130   
2  388029.pdf                                               None   
3  388030.pdf  Satellite SAR remote sensing of Great Lakes ic...   
4  388031.pdf                                     ICElecture.PDF   

         modified_date  
0  2005-08-16T11:02:56  
1  2008-05-09T12:26:32  
2  2009-01-06T16:08:54  
3  2008-01-15T07:46:14  
4  2006-08-29T23:17:50  


In [ ]:
import pandas as pd
import numpy as np
import random
import re
import time
import os

try:
    from faker import Faker

    fake = Faker()
    HAS_FAKER = True
except ImportError:
    HAS_FAKER = False

# --- CONFIGURATION ---
INPUT_RAW = "../datasets/govdocs/govdocs_metadata.csv"
INTERMEDIATE_CSV = "../datasets/govdocs/govdocs_modernized.csv"
OUTPUT_FINAL = "../datasets/govdocs/govdocs_with_paths.csv"

# Current Unix Time (Jan 1 2025)
CURRENT_YEAR_UNIX = 1735689600
TIME_SHIFT = 15 * 365 * 24 * 3600


# ==========================================
# 1. MODERNIZE & NORMALIZE
# ==========================================
def modernize_and_normalize(input_path, output_path):
    print(f"🔹 [Phase 1] Loading {input_path}...")
    df = pd.read_csv(input_path)

    # 1. FILTER: Keep only titles that look like filenames
    df = df[df["title"].str.contains(r"\.[a-zA-Z0-9]{3,4}$", na=False)].copy()
    print(f"   Processing {len(df)} rich metadata files...")

    # 2. CLEAN & MODERNIZE FILENAMES
    def clean_and_modernize_name(title):
        name = str(title).strip()
        name = re.sub(r"^Microsoft (Word|Excel|PowerPoint) - ", "", name)
        name = name.replace("\\", "/").split("/")[-1]

        base, ext = os.path.splitext(name)
        ext = ext.lower()

        # Upgrade Extensions
        if random.random() < 0.8:
            if ext == ".doc":
                ext = ".docx"
            elif ext == ".xls":
                ext = ".xlsx"
            elif ext == ".ppt":
                ext = ".pptx"
            elif ext == ".wpd":
                ext = ".docx"
            elif ext == ".ps":
                ext = ".pdf"

        # Time Travel (Update Years)
        def year_replacer(match):
            return (
                str(random.choice([2023, 2024, 2025]))
                if random.random() < 0.9
                else match.group(0)
            )

        modern_base = re.sub(r"\b(199\d|20[0-1]\d|202[0-2])\b", year_replacer, base)
        modern_base = re.sub(
            r"(FY|Q[1-4])(0[0-9]|1[0-9]|9[0-9])",
            lambda m: f"{m.group(1)}{random.choice(['23','24','25'])}",
            modern_base,
        )

        return modern_base + ext, ext

    processed = df["title"].apply(clean_and_modernize_name)
    df["filename"] = [p[0] for p in processed]
    df["extension"] = [p[1] for p in processed]

    # 3. MODERNIZE TIMESTAMPS (Cap at Today)
    df["dt"] = pd.to_datetime(df["modified_date"], errors="coerce")

    def get_modern_time(dt_val):
        if pd.isnull(dt_val):
            # Random date in last 3 years
            return CURRENT_YEAR_UNIX - random.randint(0, 3 * 365 * 86400)

        # Shift +15 years but cap at 2025
        shifted = int(dt_val.timestamp()) + TIME_SHIFT
        return min(shifted, CURRENT_YEAR_UNIX - random.randint(0, 30 * 86400))

    df["mod_time_unix"] = df["dt"].apply(get_modern_time).astype(int)

    # 4. FIX SIZES (Tiny File Fix)
    df["size_bytes"] = (df["file_size_mb"] * 1024 * 1024).astype(int)
    # If modern office file < 5KB, bump to realistic size
    mask_tiny = (df["extension"].isin([".docx", ".xlsx", ".pptx"])) & (
        df["size_bytes"] < 5000
    )
    df.loc[mask_tiny, "size_bytes"] += random.randint(15000, 50000)

    # 5. GENERATE DIVERSE USERS (Faker)
    print("   Generating diverse user pool...")
    user_pool = []
    if HAS_FAKER:
        for _ in range(100):
            user_pool.append(fake.user_name())
    else:
        # Fallback generator
        firsts = [
            "james",
            "mary",
            "john",
            "patricia",
            "robert",
            "jennifer",
            "michael",
            "linda",
            "william",
            "elizabeth",
        ]
        lasts = [
            "smith",
            "johnson",
            "williams",
            "brown",
            "jones",
            "garcia",
            "miller",
            "davis",
            "rodriguez",
        ]
        user_pool = [f"{f[0]}{l}".lower() for f in firsts for l in lasts]

    # Add Admin users
    user_pool.extend(["admin", "administrator", "root", "sysadmin", "backup_svc"])

    df["source_user"] = [random.choice(user_pool) for _ in range(len(df))]
    df["source_type"] = "govdoc_real"

    # Save
    cols = [
        "filename",
        "extension",
        "size_bytes",
        "mod_time_unix",
        "source_user",
        "source_type",
    ]
    df[cols].to_csv(output_path, index=False)
    print(f"✅ Phase 1 Done: Normalized data saved to {output_path}")


modernize_and_normalize(INPUT_RAW, INTERMEDIATE_CSV)

🔹 [Phase 1] Loading ../datasets/govdocs/govdocs_metadata.csv...
   Processing 63623 rich metadata files...
   Generating diverse user pool...
✅ Phase 1 Done: Normalized data saved to ../datasets/govdocs/govdocs_modernized.csv
🔹 [Phase 2] Loading ../datasets/govdocs/govdocs_modernized.csv...
   Generating context-aware, varied paths...
   Resolving filesystem collisions...
✅ Phase 2 Done: Final Training Data in ../datasets/govdocs/govdocs_with_paths.csv


In [12]:
def generate_paths(input_path, output_path):
    print(f"🔹 [Phase 2] Loading {input_path}...")
    df = pd.read_csv(input_path)

    # -------------------------------------------------------
    # 1. THE BRAIN: 22 CORPORATE CATEGORIES
    # -------------------------------------------------------
    CONTEXT_RULES = {
        "finance_core": {
            "keywords": [
                "budget",
                "ledger",
                "balance",
                "profit",
                "loss",
                "cashflow",
                "equity",
                "fiscal",
            ],
            "roots": [
                r"C:\Users\{user}\OneDrive - Corp\Finance\Reporting",
                r"Z:\Finance\General_Ledger\{year}",
            ],
        },
        "finance_ap_ar": {
            "keywords": [
                "invoice",
                "receipt",
                "bill",
                "payable",
                "receivable",
                "expense",
                "reimburse",
                "po",
                "purchase",
            ],
            "roots": [
                r"Z:\Accounting\AP\Invoices\{year}",
                r"C:\Users\{user}\Desktop\Expenses\Receipts_Scanned",
            ],
        },
        "finance_audit": {
            "keywords": [
                "audit",
                "compliance",
                "sox",
                "risk",
                "control",
                "fraud",
                "review",
            ],
            "roots": [
                r"Z:\Finance\Internal_Audit\{year}\Workpapers",
                r"C:\Users\{user}\Documents\Audit_Prep",
            ],
        },
        "hr_recruiting": {
            "keywords": [
                "resume",
                "cv",
                "candidate",
                "interview",
                "hiring",
                "job",
                "description",
                "recruiter",
                "screen",
            ],
            "roots": [
                r"C:\Users\{user}\Documents\HR\Talent_Acquisition\Candidates",
                r"Z:\HR\Recruiting\Reqs\{year}",
            ],
        },
        "hr_ops": {
            "keywords": [
                "employee",
                "onboarding",
                "termination",
                "handbook",
                "policy",
                "leave",
                "pto",
                "benefits",
                "insurance",
            ],
            "roots": [
                r"Z:\HR\Operations\Policies",
                r"C:\Users\{user}\OneDrive - Corp\HR_Docs\Handbooks",
            ],
        },
        "hr_comp": {
            "keywords": [
                "salary",
                "bonus",
                "commission",
                "payroll",
                "comp",
                "raise",
                "promotion",
                "equity",
                "stock",
                "vesting",
            ],
            "roots": [
                r"C:\Users\{user}\Desktop\Confidential\Comp_Planning",
                r"Z:\HR\Payroll\Cycles\{year}",
            ],
        },
        "legal_contracts": {
            "keywords": [
                "contract",
                "agreement",
                "mou",
                "sow",
                "msa",
                "terms",
                "conditions",
                "addendum",
                "renewal",
            ],
            "roots": [
                r"Z:\Legal\Contracts\Vendors\Signed",
                r"C:\Users\{user}\Documents\Legal\Drafts\Review",
            ],
        },
        "legal_litigation": {
            "keywords": [
                "suit",
                "case",
                "court",
                "litigation",
                "dispute",
                "settlement",
                "claim",
                "docket",
            ],
            "roots": [
                r"C:\Users\{user}\OneDrive - Corp\Litigation\Active_Cases",
                r"Z:\Legal\Matters\{client}",
            ],
        },
        "legal_ip": {
            "keywords": [
                "patent",
                "trademark",
                "copyright",
                "ip",
                "filing",
                "invention",
                "disclosure",
            ],
            "roots": [
                r"Z:\Legal\IP\Patents\Pending",
                r"C:\Users\{user}\Documents\IP_Filings\{year}",
            ],
        },
        "executive": {
            "keywords": [
                "strategy",
                "board",
                "minutes",
                "agenda",
                "acquisition",
                "merger",
                "investor",
                "pitch",
                "quarterly",
            ],
            "roots": [
                r"C:\Users\{user}\OneDrive - Corp\Board_Materials\{year}",
                r"C:\Users\{user}\Desktop\Strategy_2025",
            ],
        },
        "sales": {
            "keywords": [
                "proposal",
                "quote",
                "pricing",
                "crm",
                "lead",
                "prospect",
                "pipeline",
                "forecast",
                "deal",
            ],
            "roots": [
                r"C:\Users\{user}\Documents\Sales\Proposals\{client}",
                r"Z:\Sales\Pipeline_Reports\{year}",
            ],
        },
        "marketing": {
            "keywords": [
                "campaign",
                "brand",
                "logo",
                "social",
                "media",
                "press",
                "release",
                "copy",
                "blog",
                "webinar",
            ],
            "roots": [
                r"Z:\Marketing\Campaigns\{year}\Assets",
                r"C:\Users\{user}\Desktop\Creative\Drafts",
            ],
        },
        "it_dev": {
            "keywords": [
                "code",
                "script",
                "py",
                "sql",
                "java",
                "repo",
                "git",
                "api",
                "endpoint",
                "schema",
                "db",
            ],
            "roots": [
                r"C:\Users\{user}\Dev\{project}\src",
                r"C:\Users\{user}\Documents\Code_Snippets",
            ],
        },
        "it_ops": {
            "keywords": [
                "server",
                "config",
                "backup",
                "log",
                "dump",
                "error",
                "monitor",
                "patch",
                "deploy",
            ],
            "roots": [
                r"C:\Temp\Logs\Server_Dump_{year}",
                r"C:\Users\{user}\Desktop\Ops\Configs",
            ],
        },
        "it_security": {
            "keywords": [
                "vuln",
                "scan",
                "penetration",
                "threat",
                "incident",
                "key",
                "cert",
                "password",
                "auth",
            ],
            "roots": [
                r"Z:\IT\Security\Audits\{year}",
                r"C:\Users\{user}\ssh\keys\backup",
            ],
        },
        "it_architecture": {
            "keywords": [
                "spec",
                "architect",
                "diagram",
                "flow",
                "design",
                "wireframe",
                "blueprint",
            ],
            "roots": [
                r"C:\Users\{user}\Documents\Architecture\Specs",
                r"Z:\Engineering\Designs\{project}",
            ],
        },
        "r_and_d": {
            "keywords": [
                "lab",
                "experiment",
                "study",
                "research",
                "paper",
                "journal",
                "prototype",
                "test",
                "results",
            ],
            "roots": [
                r"Z:\RnD\Experiments\{year}",
                r"C:\Users\{user}\Documents\Research\Papers",
            ],
        },
        "supply_chain": {
            "keywords": [
                "logistics",
                "shipping",
                "freight",
                "inventory",
                "stock",
                "warehouse",
                "vendor",
                "supplier",
            ],
            "roots": [
                r"Z:\Ops\Supply_Chain\Manifests",
                r"C:\Users\{user}\Desktop\Inventory_Reports",
            ],
        },
        "customer_support": {
            "keywords": [
                "ticket",
                "issue",
                "complaint",
                "feedback",
                "faq",
                "guide",
                "manual",
                "help",
            ],
            "roots": [
                r"Z:\Support\Knowledge_Base\Articles",
                r"C:\Users\{user}\Downloads\Customer_Attachments",
            ],
        },
        "facilities": {
            "keywords": [
                "floorplan",
                "badge",
                "security",
                "maintenance",
                "repair",
                "lease",
                "office",
            ],
            "roots": [r"Z:\Facilities\Maps", r"C:\Users\{user}\Documents\Office_Mgmt"],
        },
        "personal": {
            "keywords": [
                "personal",
                "tax",
                "photo",
                "img",
                "scan",
                "family",
                "home",
                "insurance",
            ],
            "roots": [
                r"C:\Users\{user}\OneDrive - Corp\Personal",
                r"C:\Users\{user}\Pictures\Scans",
            ],
        },
        "general_admin": {
            "keywords": [
                "meeting",
                "notes",
                "schedule",
                "calendar",
                "memo",
                "letter",
                "fax",
                "admin",
            ],
            "roots": [
                r"C:\Users\{user}\Desktop\To_Sort",
                r"C:\Users\{user}\Documents\Misc_Work",
            ],
        },
    }

    # -------------------------------------------------------
    # 2. EXTENSION FALLBACKS (Safety Net)
    # -------------------------------------------------------
    EXTENSION_FALLBACKS = {
        ".pdf": [
            r"C:\Users\{user}\Documents\Reference\PDFs",
            r"Z:\Library\Docs",
            r"C:\Users\{user}\Downloads\Reading_List",
        ],
        ".xlsx": [
            r"C:\Users\{user}\OneDrive - Corp\Spreadsheets",
            r"C:\Users\{user}\Desktop\Data_Work",
        ],
        ".csv": [
            r"C:\Users\{user}\Documents\Data_Exports\{year}",
            r"C:\Temp\Data_Dumps",
        ],
        ".docx": [
            r"C:\Users\{user}\Documents\Drafts",
            r"C:\Users\{user}\Desktop\Writing",
        ],
        ".pptx": [
            r"C:\Users\{user}\Documents\Presentations\Decks",
            r"C:\Users\{user}\Desktop\Slides",
        ],
        ".zip": [
            r"C:\Users\{user}\Downloads\Archives",
            r"C:\Users\{user}\Desktop\To_Extract",
        ],
        ".jpg": [
            r"C:\Users\{user}\Pictures\Work_Assets",
            r"C:\Users\{user}\Desktop\Images",
        ],
        ".png": [
            r"C:\Users\{user}\Pictures\Screenshots",
            r"C:\Users\{user}\Desktop\Web_Assets",
        ],
    }

    # -------------------------------------------------------
    # 3. MASSIVE GENERAL POOL (No more "To_Sort" spam)
    # -------------------------------------------------------
    GENERIC_ROOTS = [
        r"C:\Users\{user}\Desktop\Current_Projects\{project}",
        r"C:\Users\{user}\Documents\Old_Files\Archive_{year}",
        r"C:\Users\{user}\Downloads\Web_Files",
        r"C:\Users\{user}\OneDrive - Corp\General\Shared",
        r"C:\Users\{user}\Documents\Client_Work\{client}",
        r"C:\Users\{user}\Desktop\Work_In_Progress",
        r"C:\Users\{user}\Documents\Personal\Docs",
        r"Z:\Shared\Team_Folder\General",
        r"C:\Users\{user}\Documents\Meetings\Notes_{year}",
        r"C:\Users\{user}\Desktop\Temporary",
        r"C:\Users\{user}\Documents\Scans",
        r"Z:\Public\Transfer",
        r"C:\Users\{user}\OneDrive - Corp\Backup\Laptop",
        r"C:\Users\{user}\Documents\Research\Reading_List",
        r"C:\Users\{user}\Desktop\Quick_Access",
        r"Z:\Shared\Collaboration\{project}",
        r"C:\Users\{user}\Downloads\Email_Attachments",
        r"C:\Users\{user}\Documents\Templates",
        r"C:\Users\{user}\Desktop\Review_Queue",
        r"C:\Users\{user}\OneDrive - Corp\Work\Misc",
    ]

    # Admin Specific Paths
    ADMIN_ROOTS = [
        r"C:\Users\{user}\Desktop\Scripts\Maintenance",
        r"C:\Users\{user}\Documents\Audit_Logs",
        r"Z:\IT_Admin\Tools",
        r"C:\Users\{user}\Downloads\ISOs",
    ]

    # Helpers
    CLIENTS = ["Acme", "Globex", "Soylent", "Umbrella", "Initech", "Stark", "Wayne"]
    PROJECTS = ["Alpha", "Phoenix", "Orion", "Titan", "Mercury", "Gemini"]

    # -------------------------------------------------------
    # 4. PATH GENERATION ENGINE
    # -------------------------------------------------------
    def get_context_path(row):
        fname = str(row["filename"])
        fname_lower = fname.lower()
        ext = str(row["extension"]).lower()
        user = str(row["source_user"])

        selected_template = None

        # A. Special Handling for Admin Users
        if user in ["admin", "administrator", "root", "sysadmin"]:
            if random.random() < 0.7:
                selected_template = random.choice(ADMIN_ROOTS)

        # B. Priority Check: Developer Extensions
        if not selected_template:
            if ext in [".py", ".sh", ".json", ".xml", ".java", ".cpp", ".sql"]:
                selected_template = random.choice(CONTEXT_RULES["it_dev"]["roots"])
            elif ext in [".log", ".dmp", ".config", ".ini"]:
                selected_template = random.choice(CONTEXT_RULES["it_ops"]["roots"])

        # C. Keyword Scan (The 22 Categories)
        if not selected_template:
            for cat, rules in CONTEXT_RULES.items():
                if any(k in fname_lower for k in rules["keywords"]):
                    selected_template = random.choice(rules["roots"])
                    break

        # D. Extension Fallback (If no keyword matched)
        if not selected_template and ext in EXTENSION_FALLBACKS:
            # 70% chance to use extension folder, 30% chance to go generic
            if random.random() < 0.7:
                selected_template = random.choice(EXTENSION_FALLBACKS[ext])

        # E. Generic Fallback (The "To_Sort" Killer)
        if not selected_template:
            selected_template = random.choice(GENERIC_ROOTS)

        # F. Dynamic Subfolder (Use Title Words)
        clean_name = re.sub(r"[._-]", " ", fname)
        words = [w for w in clean_name.split() if len(w) > 3 and w.isalpha()]

        subfolder = ""
        # 60% chance to add a subfolder based on the filename content
        if words and random.random() < 0.6:
            w1 = random.choice(words).capitalize()
            subfolder = f"\\{w1}"
            if len(words) > 1 and random.random() < 0.2:
                w2 = random.choice(words).capitalize()
                subfolder += f"\\{w2}"

        # G. Assemble
        path = selected_template.format(
            user=user,
            year=random.choice([2023, 2024, 2025]),
            client=random.choice(CLIENTS),
            project=random.choice(PROJECTS),
        )

        return path + subfolder

    # Apply Logic
    print("   Generating context-aware, varied paths...")
    # Using insert to put path column after filename for readability
    if "path" in df.columns:
        df = df.drop(columns=["path"])
    df.insert(1, "path", df.apply(get_context_path, axis=1))

    print("   Resolving filesystem collisions...")
    df["full_path_temp"] = df["path"] + "\\" + df["filename"]

    seen_paths = {}
    new_filenames = []

    for idx, row in df.iterrows():
        fp = row["full_path_temp"]
        fname = row["filename"]

        if fp in seen_paths:
            # Collision! Rename file.txt -> file (1).txt
            count = seen_paths[fp]
            seen_paths[fp] += 1

            if "." in fname:
                parts = fname.rsplit(".", 1)
                new_fname = f"{parts[0]} ({count}).{parts[1]}"
            else:
                new_fname = f"{fname} ({count})"
            new_filenames.append(new_fname)
        else:
            seen_paths[fp] = 1
            new_filenames.append(fname)

    df["filename"] = new_filenames
    df.drop(columns=["full_path_temp"], inplace=True, errors="ignore")

    # Save
    df.to_csv(output_path, index=False)
    print(f"✅ Phase 2 Done: Final Training Data in {output_path}")


generate_paths(INTERMEDIATE_CSV, OUTPUT_FINAL)

🔹 [Phase 2] Loading ../datasets/govdocs/govdocs_modernized.csv...
   Generating context-aware, varied paths...
   Resolving filesystem collisions...
✅ Phase 2 Done: Final Training Data in ../datasets/govdocs/govdocs_with_paths.csv


**Github code Dataset**

In [6]:
import pandas as pd
import datasets

df = pd.read_parquet(
        "../datasets/github_code/train-00000-of-01126.parquet", 
        columns=['repo_name', 'path', 'size']
    )
df.to_csv("../datasets/github_code/github_code_metadata.csv", index=False)

In [9]:
import pandas as pd
import numpy as np
import random
import re
import time
import os

try:
    from faker import Faker

    fake = Faker()
    HAS_FAKER = True
except ImportError:
    HAS_FAKER = False

# --- CONFIGURATION ---
INPUT_RAW = f"{DATASET_PATH}/github_code/github_code_metadata.csv"
INTERMEDIATE_CSV = f"{DATASET_PATH}/github_code/github_normalized_it.csv"
OUTPUT_FINAL = f"{DATASET_PATH}/github_code/github_with_paths.csv"

# STRICT ANCHOR: Jan 1, 2025 (No files created after this)
CURRENT_YEAR_UNIX = 1735689600


# ==========================================
# 1. NORMALIZE (IT Version)
# ==========================================
def prepare_github_metadata(input_path, output_path):
    print(f"🔹 [Phase 1] Loading {input_path}...")
    df = pd.read_csv(input_path)

    # 1. EXTRACT FILENAME & EXTENSION
    df["filename"] = df["path"].apply(lambda x: str(x).split("/")[-1])
    df["extension"] = df["filename"].apply(
        lambda x: "." + x.split(".")[-1].lower() if "." in x else ""
    )

    # 2. FILTER: Keep only interesting Code extensions
    TARGET_EXTS = [
        ".py",
        ".js",
        ".java",
        ".cpp",
        ".c",
        ".h",
        ".cs",
        ".php",
        ".ts",
        ".html",
        ".css",
        ".sql",
        ".json",
        ".xml",
        ".sh",
        ".go",
        ".rs",
        ".yml",
        ".yaml",
        ".dockerfile",
        ".md",
        ".rb",
        ".pl",
    ]
    df = df[df["extension"].isin(TARGET_EXTS)].copy()
    print(f"   Filtered to {len(df)} valid code files.")

    # 3. GENERATE DIVERSE USERS (Scaled Up)
    print("   Generating developer personas...")
    user_pool = []

    # A. Faker Users (Pool size increased to 500 for variety)
    if HAS_FAKER:
        for _ in range(500):
            user_pool.append(fake.user_name())
    else:
        # Fallback
        base = ["dev", "coder", "admin", "test", "user"]
        user_pool = [f"{b}{i}" for b in base for i in range(100)]

    # B. IT Specific Roles
    it_roles = [
        "admin",
        "root",
        "sysadmin",
        "deploy",
        "gitlab_runner",
        "jenkins_bot",
        "db_admin",
        "devops",
        "cloud_user",
        "aws_svc",
        "azure_agent",
        "backup_svc",
    ]

    # Assign Users (85% regular devs, 15% infra/admin accounts)
    df["source_user"] = [
        random.choice(user_pool) if random.random() > 0.15 else random.choice(it_roles)
        for _ in range(len(df))
    ]

    # 4. GENERATE TIMESTAMPS (Strict Past)
    # Generate dates strictly between Jan 2023 and Jan 1, 2025
    two_years_sec = 2 * 365 * 24 * 3600

    def get_past_time():
        return CURRENT_YEAR_UNIX - random.randint(0, two_years_sec)

    df["mod_time_unix"] = [get_past_time() for _ in range(len(df))]

    # 5. FINALIZE SCHEMA
    df["size_bytes"] = df["size"]
    df["source_type"] = "github_real"

    # Keep internal path for Phase 2
    df.rename(columns={"path": "internal_path"}, inplace=True)

    cols = [
        "filename",
        "extension",
        "size_bytes",
        "mod_time_unix",
        "source_user",
        "source_type",
        "repo_name",
        "internal_path",
    ]
    df[cols].to_csv(output_path, index=False)
    print(f"✅ Phase 1 Done: Normalized IT data saved to {output_path}")


# ==========================================
# 2. GENERATE COMPLEX IT PATHS
# ==========================================
def generate_it_paths(input_path, output_path):
    print(f"🔹 [Phase 2] Loading {input_path}...")
    df = pd.read_csv(input_path)

    # -------------------------------------------------------
    # 1. THE BRAIN: IT ENVIRONMENT CATEGORIES
    # -------------------------------------------------------
    IT_CONTEXTS = {
        "local_dev_windows": {
            "roots": [
                r"C:\Users\{user}\source\repos\{repo}",
                r"C:\Users\{user}\Dev\{repo}",
                r"C:\Users\{user}\Documents\GitHub\{repo}",
                r"C:\Work\Projects\{repo}",
                r"C:\Users\{user}\Desktop\Coding\{repo}",
            ]
        },
        "local_dev_linux_wsl": {
            "roots": [
                r"\\wsl$\Ubuntu\home\{user}\projects\{repo}",
                r"\\wsl$\Debian\home\{user}\git\{repo}",
                r"C:\Users\{user}\AppData\Local\Packages\Canonical\rootfs\home\{user}\{repo}",
            ]
        },
        "server_production": {
            "roots": [
                r"C:\inetpub\wwwroot\{repo}",
                r"C:\Apps\Production\{repo}",
                r"D:\Websites\{repo}\public_html",
                r"C:\Program Files\Enterprise Apps\{repo}",
            ]
        },
        "ci_cd_builds": {
            "roots": [
                r"C:\Jenkins\workspace\{repo}_Build",
                r"C:\GitLab-Runner\builds\{hash}\{repo}",
                r"D:\Build_Artifacts\{repo}\{version}",
                r"C:\Temp\Publish\{repo}",
            ]
        },
        "backups_archives": {
            "roots": [
                r"Z:\Backups\Source_Code\{repo}_{year}",
                r"D:\Archive\Legacy_Projects\{repo}",
                r"C:\Users\{user}\OneDrive - Corp\Backups\Code\{repo}",
            ]
        },
        "temp_review": {
            "roots": [
                r"C:\Users\{user}\Downloads\{repo}-master",
                r"C:\Users\{user}\Desktop\Code_Review\{repo}",
                r"C:\Users\{user}\AppData\Local\Temp\git\{repo}",
            ]
        },
        "admin_tools": {
            "roots": [
                r"C:\Users\{user}\Desktop\Scripts\Maintenance",
                r"Z:\IT_Admin\Tools\{repo}",
                r"C:\Users\{user}\Documents\Audit_Logs\Scripts",
            ]
        },
    }

    # -------------------------------------------------------
    # 2. PATH GENERATION ENGINE
    # -------------------------------------------------------
    def get_it_path(row):
        user = str(row["source_user"])
        repo_raw = str(row["repo_name"])

        # Clean repo name
        repo = repo_raw.split("/")[-1] if "/" in repo_raw else repo_raw

        # Clean internal path
        internal_path = str(row["internal_path"]).replace("/", "\\")
        if "\\" in internal_path:
            folder_structure = internal_path.rsplit("\\", 1)[0]
        else:
            folder_structure = ""

        # A. Determine Context based on User Role
        # Admins get specific paths
        if user in ["admin", "root", "sysadmin", "backup_svc"]:
            context = random.choice(
                ["server_production", "backups_archives", "admin_tools"]
            )
        elif user in ["deploy", "jenkins_bot", "gitlab_runner"]:
            context = "ci_cd_builds"
        else:
            # Regular devs
            dice = random.random()
            if dice < 0.60:
                context = "local_dev_windows"
            elif dice < 0.80:
                context = "local_dev_linux_wsl"
            elif dice < 0.95:
                context = "temp_review"
            else:
                context = "server_production"  # Dev testing in prod (oops)

        # B. Select Template
        template = random.choice(IT_CONTEXTS[context]["roots"])

        # C. Fill Template
        root_path = template.format(
            user=user,
            repo=repo,
            year=random.choice(
                [2023, 2024]
            ),  # No 2025 to avoid future conflicts in archives
            hash=random.randint(100000, 999999),
            version=f"v{random.randint(1,9)}.{random.randint(0,5)}",
        )

        # D. Combine
        if folder_structure:
            return f"{root_path}\\{folder_structure}"
        else:
            return root_path

    print("   Generating IT ecosystem paths...")
    # Safe insert
    if "path" in df.columns:
        df.drop(columns=["path"], inplace=True)
    df.insert(1, "path", df.apply(get_it_path, axis=1))

    # -------------------------------------------------------
    # 3. FINAL SANITY CHECK: RESOLVE COLLISIONS
    # -------------------------------------------------------
    print("   Resolving filesystem collisions...")
    df["full_path_temp"] = df["path"] + "\\" + df["filename"]

    seen_paths = {}
    new_filenames = []

    for idx, row in df.iterrows():
        fp = row["full_path_temp"]
        fname = row["filename"]

        if fp in seen_paths:
            count = seen_paths[fp]
            seen_paths[fp] += 1

            # Rename: config.json -> config (1).json
            if "." in fname:
                parts = fname.rsplit(".", 1)
                new_fname = f"{parts[0]} ({count}).{parts[1]}"
            else:
                new_fname = f"{fname} ({count})"
            new_filenames.append(new_fname)
        else:
            seen_paths[fp] = 1
            new_filenames.append(fname)

    df["filename"] = new_filenames
    df.drop(
        columns=["full_path_temp", "repo_name", "internal_path"],
        inplace=True,
        errors="ignore",
    )

    # Save
    df.to_csv(output_path, index=False)
    print(f"✅ Phase 2 Done: Final IT Training Data in {output_path}")


prepare_github_metadata(INPUT_RAW, INTERMEDIATE_CSV)
generate_it_paths(INTERMEDIATE_CSV, OUTPUT_FINAL)

🔹 [Phase 1] Loading ../datasets/github_code/github_code_metadata.csv...
   Filtered to 96010 valid code files.
   Generating developer personas...
✅ Phase 1 Done: Normalized IT data saved to ../datasets/github_code/github_normalized_it.csv
🔹 [Phase 2] Loading ../datasets/github_code/github_normalized_it.csv...
   Generating IT ecosystem paths...
   Resolving filesystem collisions...
✅ Phase 2 Done: Final IT Training Data in ../datasets/github_code/github_with_paths.csv


In [10]:
df = pd.read_csv(f"{DATASET_PATH}/github_code/github_with_paths.csv")
df[:1000].to_csv(f"{DATASET_PATH}/github_code/github_sample_1k.csv", index=False)